# NB-204: Data Loading & Processing

Efficient data loading for LLM training.

## Learning Objectives
- Dataset creation
- Data tokenization
- DataLoader optimization
- Streaming datasets

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from datasets import load_dataset

print("Data loading ready")

## 1. Custom Dataset

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=512):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # TODO: Tokenize
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
        }

# Test
tokenizer = AutoTokenizer.from_pretrained("gpt2")
texts = ["Hello world", "This is a test"] * 10

dataset = TextDataset(texts, tokenizer, max_length=32)
print(f"Dataset size: {len(dataset)}")
print(f"Sample: {dataset[0]['input_ids'].shape}")

## 2. DataLoader with Collate

In [ ]:
def collate_fn(batch):
    """Custom collate function for dynamic padding."""

    # TODO: Find max length in batch
    max_len = max(item["input_ids"].size(0) for item in batch)

    # Pad to max length
    input_ids = torch.zeros(len(batch), max_len, dtype=torch.long)
    attention_mask = torch.zeros(len(batch), max_len, dtype=torch.long)

    for i, item in enumerate(batch):
        seq_len = item["input_ids"].size(0)
        input_ids[i, :seq_len] = item["input_ids"]
        attention_mask[i, :seq_len] = item["attention_mask"]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
    }

# Create DataLoader
dataloader = DataLoader(dataset, batch_size=4, collate_fn=collate_fn)

batch = next(iter(dataloader))
print(f"Batch shape: {batch['input_ids'].shape}")

## 3. Streaming Datasets

In [ ]:
# TODO: Load streaming dataset
# dataset = load_dataset("c4", "en", streaming=True)

# Take first 1000 examples
# train_data = dataset["train"].take(1000)

print("Streaming datasets require HuggingFace datasets library")
print("Use: dataset = load_dataset('dataset_name', streaming=True)")

## 4. Data Augmentation

In [ ]:
def augment_text(text, augments=[]):
    """Apply text augmentations."""

    augmented = text

    # TODO: Apply augmentations
    for aug in augments:
        if aug == "uppercase_first":
            augmented = augmented[0].upper() + augmented[1:]
        elif aug == "add_prefix":
            augmented = "Please: " + augmented
        elif aug == "lowercase":
            augmented = augmented.lower()

    return augmented

# Test
text = "hello world"
augmented = augment_text(text, ["uppercase_first", "add_prefix"])
print(f"Original: {text}")
print(f"Augmented: {augmented}")

## Exercise

1. Implement masked language modeling dataset
2. Add multiprocess data loading
3. Implement dynamic batching